# Credit Risk Scoring System Using Multivariate Statistical Analysis and Machine Learning

## Data Understanding

### Project Overview

Credit risk assessment is a critical task in the Banking, Financial Services, and Insurance (BFSI) sector. Financial institutions use applicant information such as demographic details, financial history, credit records, and repayment behavior to determine the likelihood of loan default.

This project aims to develop an interpretable Credit Risk Scoring System using statistical analysis, Principal Component Analysis (PCA), and Machine Learning techniques on the Home Credit Default Risk dataset.

This notebook focuses on understanding the datasets before any preprocessing or modeling is performed.

---

## Objectives

- Understand the structure of each dataset.
- Identify relationships between datasets.
- Explore primary and foreign keys.
- Analyze the target variable.
- Examine data types and feature categories.
- Summarize missing values.
- Prepare for the preprocessing phase.

---

# Import Required Libraries

The following libraries are used for data loading, manipulation, visualization, and basic exploratory analysis.

In [1]:
# ============================================================
# Import Libraries
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# ------------------------------------------------------------
# Display Settings
# ------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

# ------------------------------------------------------------
# Visualization Settings
# ------------------------------------------------------------

plt.style.use("ggplot")

sns.set_theme(style="whitegrid")

# ------------------------------------------------------------
# Random Seed
# ------------------------------------------------------------

RANDOM_STATE = 42

# Dataset Directory

All datasets are stored inside the **data/** directory.

The Home Credit Default Risk dataset consists of multiple relational tables linked through unique identifiers such as **SK_ID_CURR**, **SK_ID_PREV**, and **SK_ID_BUREAU**.

In [2]:
# ============================================================
# Define Dataset Directory
# ============================================================

DATA_DIR = Path("../data")

# Load Datasets

The project uses multiple datasets provided by the Home Credit Default Risk competition.

Each dataset captures a different aspect of an applicant's financial history.

The datasets will be loaded individually and explored before being integrated during the preprocessing phase.

In [3]:
# ============================================================
# Load Datasets
# ============================================================

application_df = pd.read_csv(DATA_DIR / "HC_application_train.csv")

bureau_df = pd.read_csv(DATA_DIR / "HC_bureau.csv")

bureau_balance_df = pd.read_csv(DATA_DIR / "HC_bureau_balance.csv")

previous_application_df = pd.read_csv(DATA_DIR / "HC_previous_application.csv")

installments_df = pd.read_csv(DATA_DIR / "HC_installments_payments.csv")

credit_card_df = pd.read_csv(DATA_DIR / "HC_credit_card_balance.csv")

pos_cash_df = pd.read_csv(DATA_DIR / "HC_POS_CASH_balance.csv")

#columns_description_df = pd.read_csv(DATA_DIR / "HomeCredit_columns_description.csv", encoding="latin1")

# Verify Dataset Loading

Before performing any analysis, verify that all datasets have been successfully loaded.

The following section displays the dimensions of each dataset.

In [4]:
# ============================================================
# Dataset Shapes
# ============================================================

datasets = {
    "Application Train": application_df,
    "Bureau": bureau_df,
    "Bureau Balance": bureau_balance_df,
    "Previous Applications": previous_application_df,
    "Installments Payments": installments_df,
    "Credit Card Balance": credit_card_df,
    "POS Cash Balance": pos_cash_df,
    #"Column Description": columns_description_df
}

shape_summary = pd.DataFrame(
    {
        "Dataset": datasets.keys(),
        "Rows": [df.shape[0] for df in datasets.values()],
        "Columns": [df.shape[1] for df in datasets.values()]
    }
)

shape_summary

,Dataset,Rows,Columns
0,Application Train,307511,122
1,Bureau,1716428,17
2,Bureau Balance,27299925,3
3,Previous Applications,1670214,37
4,Installments Payments,13605401,8
5,Credit Card Balance,3840312,23
6,POS Cash Balance,10001358,8


# Dataset Summary

Before exploring each dataset individually, it is useful to create reusable functions that summarize the structure and quality of the data.

The following summary includes:

- Number of rows and columns
- Memory usage
- Number of missing values
- Duplicate records
- Data types
- Numerical and categorical feature counts

These summaries provide a quick understanding of each dataset and help identify potential preprocessing challenges.

In [5]:
# ============================================================
# Dataset Summary Function
# ============================================================

def summarize_dataset(df, dataset_name):
    """
    Generate a summary of a dataset.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset to summarize.

    dataset_name : str
        Name of the dataset.

    Returns
    -------
    pandas.DataFrame
        Dataset summary.
    """

    summary = pd.DataFrame({
        "Dataset": [dataset_name],
        "Rows": [df.shape[0]],
        "Columns": [df.shape[1]],
        "Missing Values": [df.isnull().sum().sum()],
        "Duplicate Rows": [df.duplicated().sum()],
        "Memory Usage (MB)": [round(df.memory_usage(deep=True).sum() / 1024**2, 2)],
        "Numerical Features": [len(df.select_dtypes(include=np.number).columns)],
        "Categorical Features": [len(df.select_dtypes(include="object").columns)]
    })

    return summary

In [6]:
# ============================================================
# Dataset Summaries
# ============================================================

summary_tables = []

for dataset_name, dataframe in datasets.items():
    summary_tables.append(
        summarize_dataset(dataframe, dataset_name)
    )

dataset_summary = pd.concat(summary_tables, ignore_index=True)

dataset_summary

,Dataset,Rows,Columns,Missing Values,Duplicate Rows,Memory Usage (MB),Numerical Features,Categorical Features
0,Application Train,307511,122,9152465,0,325.22,106,16
1,Bureau,1716428,17,3939947,0,271.48,14,3
2,Bureau Balance,27299925,3,0,0,650.88,2,1
3,Previous Applications,1670214,37,11109336,0,671.81,21,16
4,Installments Payments,13605401,8,5810,0,830.41,8,0
5,Credit Card Balance,3840312,23,5877356,0,696.23,22,1
6,POS Cash Balance,10001358,8,52158,0,669.89,7,1


# Display Basic Dataset Information

After obtaining a high-level summary, the next step is to inspect the structure of each dataset.

For every dataset, we will examine:

- First five records
- Column names
- Data types
- Missing values
- Memory usage

This helps us understand the schema before preprocessing.

In [7]:
# ============================================================
# Dataset Information Function
# ============================================================

def inspect_dataset(df, dataset_name, n_rows=5):
    """
    Display basic information about a dataset.

    Parameters
    ----------
    df : pandas.DataFrame
    dataset_name : str
    n_rows : int
    """

    print("=" * 80)
    print(f"{dataset_name.upper()}")
    print("=" * 80)

    print(f"\nShape : {df.shape}")

    print("\nFirst Records")
    display(df.head(n_rows))

    print("\nColumn Names")
    display(pd.DataFrame(df.columns, columns=["Column Name"]))

    print("\nData Types")
    display(df.dtypes.reset_index().rename(
        columns={
            "index": "Column",
            0: "Data Type"
        }
    ))

In [8]:
#inspect_dataset(application_df, "Application Train")
#inspect_dataset(bureau_df, "Bureau")
#inspect_dataset(bureau_balance_df, "Bureau Balance")
#inspect_dataset(previous_application_df, "Previous Applications")
#inspect_dataset(installments_df, "Installments Payments")
#inspect_dataset(credit_card_df, "Credit Card Balance")
#inspect_dataset(pos_cash_df, "POS Cash Balance")
#inspect_dataset(columns_description_df, "Column Description")

# Missing Value Analysis

Missing values are common in real-world financial datasets.

Understanding the extent of missing data helps determine appropriate preprocessing strategies such as:

- Removal
- Mean or median imputation
- Mode imputation
- Model-based imputation

The following function summarizes missing values for every feature.

In [9]:
# ============================================================
# Missing Value Summary
# ============================================================

def missing_value_summary(df):

    missing = pd.DataFrame({
        "Missing Count": df.isnull().sum(),
        "Missing Percentage":
        (df.isnull().mean() * 100).round(2)
    })

    missing = missing[missing["Missing Count"] > 0]

    return missing.sort_values(
        by="Missing Percentage",
        ascending=False
    )

In [10]:
#missing_value_summary(application_df).head(20)
#missing_value_summary(bureau_df).head(20)
#missing_value_summary(bureau_balance_df).head(20)
#missing_value_summary(previous_application_df).head(20)
#missing_value_summary(installments_df).head(20)
#missing_value_summary(credit_card_df).head(20)
#missing_value_summary(pos_cash_df).head(20)
#missing_value_summary(columns_description_df).head(20)

# Statistical Summary

The statistical summary provides descriptive measures for numerical variables, including:

- Mean
- Standard Deviation
- Minimum
- Quartiles
- Maximum

These statistics help identify unusual distributions and potential outliers before visualization.

In [11]:
#application_df.describe().T
#bureau_df.describe().T
#bureau_balance_df.describe().T
#previous_application_df.describe().T
#installments_df.describe().T
#credit_card_df.describe().T
#pos_cash_df.describe().T
#columns_description_df.describe().T

# Initial Observations

Based on the preliminary inspection of the Home Credit Default Risk datasets, the following observations were made:

## 1. Dataset Dimensions

The project consists of seven interconnected datasets with varying sizes. The **Application Train** dataset serves as the primary table containing applicant information and the target variable (`TARGET`), while the remaining datasets provide historical financial records.

| Dataset | Rows | Columns |
|---------|------:|--------:|
| Application Train | 307,511 | 122 |
| Bureau | 1,716,428 | 17 |
| Bureau Balance | 27,299,925 | 3 |
| Previous Applications | 1,670,214 | 37 |
| Installments Payments | 13,605,401 | 8 |
| Credit Card Balance | 3,840,312 | 23 |
| POS Cash Balance | 10,001,358 | 8 |

**Observation:**

- The auxiliary datasets are substantially larger than the primary application dataset because they contain multiple historical records for each applicant.
- These datasets cannot be merged directly and will require aggregation at the applicant level (`SK_ID_CURR`) during preprocessing.

---

## 2. Missing Values

Several datasets contain a significant number of missing values.

| Dataset | Missing Values |
|---------|---------------:|
| Application Train | 9,152,465 |
| Bureau | 3,939,947 |
| Bureau Balance | 0 |
| Previous Applications | 11,109,336 |
| Installments Payments | 5,810 |
| Credit Card Balance | 5,877,356 |
| POS Cash Balance | 52,158 |

**Observation:**

- Missing values are a major characteristic of this dataset.
- The **Previous Applications** and **Application Train** datasets contain the highest number of missing entries.
- In contrast, **Bureau Balance** has no missing values, making it the cleanest dataset.
- A feature-wise missing value analysis will be required before deciding whether to impute or remove variables.

---

## 3. Feature Types

### Categorical Features

| Dataset | Count |
|---------|------:|
| Application Train | 16 |
| Bureau | 3 |
| Bureau Balance | 1 |
| Previous Applications | 16 |
| Installments Payments | 0 |
| Credit Card Balance | 1 |
| POS Cash Balance | 1 |

### Numerical Features

| Dataset | Count |
|---------|------:|
| Application Train | 106 |
| Bureau | 14 |
| Bureau Balance | 2 |
| Previous Applications | 21 |
| Installments Payments | 8 |
| Credit Card Balance | 22 |
| POS Cash Balance | 7 |

**Observation:**

- The datasets are predominantly numerical, making them well suited for statistical analysis and machine learning.
- Categorical variables will require appropriate encoding techniques before model training.
- Numerical variables will require scaling prior to applying Principal Component Analysis (PCA).

---

## 4. Data Quality Assessment

The initial inspection indicates that the datasets are generally well structured, with no immediate issues related to schema consistency. However, several preprocessing challenges must be addressed:

- Large numbers of missing values across multiple datasets.
- Multiple historical records per applicant requiring aggregation.
- High-dimensional feature space that may introduce multicollinearity.
- Presence of categorical variables requiring encoding.
- Potential outliers in financial variables that will require further investigation during Exploratory Data Analysis (EDA).

---

## 5. Implications for the Next Phase

Based on these observations, the preprocessing phase will include the following tasks:

- Analyze missing values at the feature level.
- Remove duplicate records, if any.
- Aggregate transactional datasets using `SK_ID_CURR`.
- Merge all datasets into a unified applicant-level dataset.
- Encode categorical variables.
- Handle missing values using appropriate imputation strategies.
- Scale numerical features.
- Prepare the dataset for Exploratory Data Analysis, statistical hypothesis testing, Principal Component Analysis (PCA), and machine learning.